In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG crypto_pipeline")

DataFrame[]

In [0]:
# Databricks notebook source

# ==========================================================
# Silver Layer
# Notebook : 02_build_fact_coin_price
#
# Purpose:
# Build fact_coin_price
#
# Grain:
# One row per Coin per Observation Timestamp
#
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "crypto_pipeline"

BRONZE_TABLE = f"{CATALOG}.bronze.raw_coin_market_data"
DIM_TABLE = f"{CATALOG}.silver.dim_coin"
FACT_TABLE = f"{CATALOG}.silver.fact_coin_price"

spark.sql(f"USE CATALOG {CATALOG}")

# ==========================================================
# Create Fact Table
# ==========================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {FACT_TABLE}

(

coin_sk STRING,

coin_id STRING,

observation_ts TIMESTAMP,

current_price DOUBLE,

market_cap DOUBLE,

total_volume DOUBLE,

market_cap_rank INT,

price_change_24h DOUBLE,

price_change_percentage_24h DOUBLE

)

USING DELTA

""")

# ==========================================================
# Read Bronze
# ==========================================================

bronze_df = spark.table(BRONZE_TABLE)

# ==========================================================
# Read Current Dimension
# ==========================================================

dim_df = (

    spark.table(DIM_TABLE)

    .filter("is_current = true")

)

# ==========================================================
# Resolve Surrogate Key
# ==========================================================

fact_stage = (

    bronze_df.alias("b")

    .join(

        dim_df.alias("d"),

        F.col("b.id") == F.col("d.coin_id"),

        "inner"

    )

    .select(

        F.col("d.coin_sk"),

        F.col("b.id").alias("coin_id"),

        F.col("b._ingested_at").alias("observation_ts"),

        F.col("b.current_price"),

        F.col("b.market_cap"),

        F.col("b.total_volume"),

        F.col("b.market_cap_rank"),

        F.col("b.price_change_24h"),

        F.col("b.price_change_percentage_24h")

    )

)

print("Fact Staging")

display(fact_stage)

print("Rows:", fact_stage.count())

Fact Staging


coin_sk,coin_id,observation_ts,current_price,market_cap,total_volume,market_cap_rank,price_change_24h,price_change_percentage_24h
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,2026-07-22T08:58:57.519Z,65922.0,1.322409077378E12,3.2062806671E10,1,-251.1924731802137,-0.3796
ae9869a81d32f512573269df55d19b5b46ed939fbe66e81b3755be68304c9406,ethereum,2026-07-22T08:58:57.519Z,1919.35,2.31629638606E11,1.0336311395E10,2,-18.99310764918164,-0.97986
3f869d3dfc0afcb89b92b3278a6eb938b533dd3167e83ddeb6088fa2940b651f,tether,2026-07-22T08:58:57.519Z,0.999272,1.84091192562E11,4.8507675246E10,3,8.548E-5,0.00855
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,binancecoin,2026-07-22T08:58:57.519Z,569.71,7.587212412E10,5.64901171E8,4,-7.535549706710867,-1.30542
8b1c69f728a102671afb803a01c594274da853415b5fa294ccb5ebd10a916aea,usd-coin,2026-07-22T08:58:57.519Z,1.0,7.3198530272E10,1.1191746137E10,5,3.4142E-4,0.03415
15693b928f9f1c8df421b733df7d6adbcafded5444e309cecdac7a7510520edc,ripple,2026-07-22T08:58:57.519Z,1.13,7.0829003636E10,1.35471518E9,6,9.2615E-4,0.08175
5279baec1352bf208376bcf4cd0423ed00708e142187575e0cddd0b50393d34f,solana,2026-07-22T08:58:57.519Z,77.34,4.507139695E10,1.496050329E9,7,-0.9703418514234698,-1.23907
e2397c39195878c724b52c41f7b5e9587bb0e7a6e4fdbc3d2cc4e1f7aaefb3ba,tron,2026-07-22T08:58:57.519Z,0.328866,3.1200183732E10,3.25557615E8,8,0.0023698,0.72583
95b12ebb7f9edf3d7b5a9a3faef2e33aa0db75684755729b1d7938c11214fc6c,figure-heloc,2026-07-22T08:58:57.519Z,1.005,2.0314058647E10,5.5088452E7,9,0.0047664,0.47635
8ff3179a3e298e8b1a9a01812be6cfc086087b1adf0355bcbd98fc4432edea36,hyperliquid,2026-07-22T08:58:57.519Z,58.79,1.3076702209E10,4.10973627E8,10,-4.197472609852049,-6.66412


Rows: 100


In [0]:
window = Window.partitionBy(

    "coin_id",

    "observation_ts"

).orderBy(

    F.col("market_cap").desc()

)

fact_stage = (

    fact_stage

    .withColumn(

        "rn",

        F.row_number().over(window)

    )

    .filter("rn=1")

    .drop("rn")

)

print("Duplicates Removed")

display(fact_stage)

Duplicates Removed


coin_sk,coin_id,observation_ts,current_price,market_cap,total_volume,market_cap_rank,price_change_24h,price_change_percentage_24h
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,aster-2,2026-07-22T08:58:57.519Z,0.622088,1.670620127E9,6.3981992E7,46,-0.00839127419145902,-1.33094
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,aster-2,2026-07-22T08:59:21.134Z,0.622088,1.670620127E9,6.3981992E7,46,-0.00839127419145902,-1.33094
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,2026-07-22T08:58:57.519Z,6.51,2.812036807E9,1.21168985E8,32,-0.14037606851970175,-2.10982
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,2026-07-22T08:59:21.134Z,6.51,2.812036807E9,1.21168985E8,32,-0.14037606851970175,-2.10982
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,binancecoin,2026-07-22T08:58:57.519Z,569.71,7.587212412E10,5.64901171E8,4,-7.535549706710867,-1.30542
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,binancecoin,2026-07-22T08:59:21.134Z,569.71,7.587212412E10,5.64901171E8,4,-7.535549706710867,-1.30542
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,2026-07-22T08:58:57.519Z,65922.0,1.322409077378E12,3.2062806671E10,1,-251.1924731802137,-0.3796
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,2026-07-22T08:59:21.134Z,65922.0,1.322409077378E12,3.2062806671E10,1,-251.1924731802137,-0.3796
c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,bitcoin-cash,2026-07-22T08:58:57.519Z,220.78,4.430395854E9,8.4996954E7,23,-2.645743369264551,-1.1842
c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,bitcoin-cash,2026-07-22T08:59:21.134Z,220.78,4.430395854E9,8.4996954E7,23,-2.645743369264551,-1.1842


In [0]:
print("Distinct Coins")

print(

    fact_stage

    .select("coin_id")

    .distinct()

    .count()

)

print("Observation Rows")

print(

    fact_stage.count()

)

Distinct Coins
50
Observation Rows
100


In [0]:
# ==========================================================
# PART 2
# MERGE INTO FACT TABLE
# ==========================================================

from delta.tables import DeltaTable

fact_delta = DeltaTable.forName(spark, FACT_TABLE)

(
    fact_delta.alias("target")
    .merge(
        fact_stage.alias("source"),
        """
        target.coin_id = source.coin_id
        AND target.observation_ts = source.observation_ts
        """
    )
    .whenNotMatchedInsert(
        values={
            "coin_sk": "source.coin_sk",
            "coin_id": "source.coin_id",
            "observation_ts": "source.observation_ts",
            "current_price": "source.current_price",
            "market_cap": "source.market_cap",
            "total_volume": "source.total_volume",
            "market_cap_rank": "source.market_cap_rank",
            "price_change_24h": "source.price_change_24h",
            "price_change_percentage_24h":
                "source.price_change_percentage_24h"
        }
    )
    .execute()
)

print("Fact MERGE Complete")

Fact MERGE Complete


In [0]:
fact_df = spark.table(FACT_TABLE)

display(
    fact_df.orderBy(
        F.desc("observation_ts"),
        "coin_id"
    )
)

print("Total Rows:", fact_df.count())

coin_sk,coin_id,observation_ts,current_price,market_cap,total_volume,market_cap_rank,price_change_24h,price_change_percentage_24h
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,aster-2,2026-07-22T08:59:21.134Z,0.622088,1.670620127E9,6.3981992E7,46,-0.00839127419145902,-1.33094
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,2026-07-22T08:59:21.134Z,6.51,2.812036807E9,1.21168985E8,32,-0.14037606851970175,-2.10982
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,binancecoin,2026-07-22T08:59:21.134Z,569.71,7.587212412E10,5.64901171E8,4,-7.535549706710867,-1.30542
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,2026-07-22T08:59:21.134Z,65922.0,1.322409077378E12,3.2062806671E10,1,-251.1924731802137,-0.3796
c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,bitcoin-cash,2026-07-22T08:59:21.134Z,220.78,4.430395854E9,8.4996954E7,23,-2.645743369264551,-1.1842
e44988b4e78b3b5f96854b7d376c9a1e5dddba8476196e5dd3000284a1488c5c,bittensor,2026-07-22T08:59:21.134Z,196.33,1.884358244E9,9.7337906E7,42,-2.3767942079357454,-1.19614
e7735b4253b64a49c688af4708daa3dd47c829fd075ff74ea83fff1b99246733,blackrock-usd-institutional-digital-liquidity-fund,2026-07-22T08:59:21.134Z,1.0,2.536756529E9,0.0,35,0.0,0.0
126bda23e48f287091cc02ebdc4a972096a6bc774223da74dc40c1ab85f18249,canton-network,2026-07-22T08:59:21.134Z,0.124072,4.853486737E9,8252504.0,21,-0.001422113417409582,-1.13321
6235bef37881100ed67526ae2a1af07ab4725e2e7b1f7ae709c29e01a6bc53e2,cardano,2026-07-22T08:59:21.134Z,0.172065,6.413681221E9,2.82280937E8,20,-0.002502654314636171,-1.43363
b76527ae1b44a61572d49f1ac34393c9d4b2acc91346d9bdc978b09fe711be00,chainlink,2026-07-22T08:59:21.134Z,8.6,6.437886561E9,1.7999511E8,19,-0.10505516988922459,-1.20617


Total Rows: 100


In [0]:
duplicates = (

    fact_df

    .groupBy(

        "coin_id",

        "observation_ts"

    )

    .count()

    .filter("count > 1")

)

display(duplicates)

print("Duplicate Rows:", duplicates.count())

coin_id,observation_ts,count


Duplicate Rows: 0


In [0]:
print(

    fact_df

    .select("coin_id")

    .distinct()

    .count()

)

50


In [0]:
display(

    fact_df

    .orderBy(

        F.desc("observation_ts")

    )

)

coin_sk,coin_id,observation_ts,current_price,market_cap,total_volume,market_cap_rank,price_change_24h,price_change_percentage_24h
d6bfd0a6475c30d62312b53074b22a31b6debebf6d93f2cae922225f03643afe,global-dollar,2026-07-22T08:59:21.134Z,0.999781,3.250460653E9,2.18795318E8,28,-3.86593896118836E-4,-0.03865
e44988b4e78b3b5f96854b7d376c9a1e5dddba8476196e5dd3000284a1488c5c,bittensor,2026-07-22T08:59:21.134Z,196.33,1.884358244E9,9.7337906E7,42,-2.3767942079357454,-1.19614
8ff3179a3e298e8b1a9a01812be6cfc086087b1adf0355bcbd98fc4432edea36,hyperliquid,2026-07-22T08:59:21.134Z,58.79,1.3076702209E10,4.10973627E8,10,-4.197472609852049,-6.66412
89a42e9565784d167dc7ad5e50f91d45b6aa3141326e55c332043b9b322a9a3f,shiba-inu,2026-07-22T08:59:21.134Z,4.23E-6,2.491836453E9,4.3292614E7,37,-5.256455563E-8,-1.22785
ae9869a81d32f512573269df55d19b5b46ed939fbe66e81b3755be68304c9406,ethereum,2026-07-22T08:59:21.134Z,1919.35,2.31629638606E11,1.0336311395E10,2,-18.99310764918164,-0.97986
23fec56b98d88017bcab27610360b3ed3e1ad86e04aed9093989d62cb6090b03,near,2026-07-22T08:59:21.134Z,1.88,2.44767747E9,2.02695092E8,38,-0.1251788552169344,-6.24059
7aee56260d91a83514535669eb91b1d90b5bf21904882ed36544ba98ce361984,litecoin,2026-07-22T08:59:21.134Z,46.4,3.592415315E9,2.23993187E8,27,-1.2523275377008716,-2.62785
fa30dc7298e987e69d2011d03ae7759fa16116678fbb21f7aa50a199117cbd3e,uniswap,2026-07-22T08:59:21.134Z,3.72,2.323478229E9,1.3652852E8,39,4.1788E-4,0.01124
3f869d3dfc0afcb89b92b3278a6eb938b533dd3167e83ddeb6088fa2940b651f,tether,2026-07-22T08:59:21.134Z,0.999272,1.84091192562E11,4.8507675246E10,3,8.548E-5,0.00855
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,2026-07-22T08:59:21.134Z,65922.0,1.322409077378E12,3.2062806671E10,1,-251.1924731802137,-0.3796
